# The Encoder-Decoder Pipeline — A Real Data Engineering Problem

You are handed a real task at work:

> We bought a **recommendation engine**. It reads a CSV file with three columns —
> `user_id, product_id, score` — and writes its recommendations to another CSV
> with columns `user_id, product_id, recommendation_score`.
> **It only accepts integers** for `user_id`.
>
> Our data, however, identifies users by **email address**:
>
> ```
> sandeep,1, 1.0
> nitin, 2, 3.0
> subodh, 2, 1.0
> sandeep, 2, 2.0
> nitin, 1, 2.0
> ```
>
> Make it work. And when the engine answers, translate its output **back** to emails.

You cannot change the engine. So you will wrap it:

```
our data (emails)                                      final output (emails)
      │                                                        ▲
      ▼                                                        │
  ENCODER ──→ engine input (ints) ──→ RECOMMENDER ──→ engine output (ints) ──→ DECODER
```

The Encoder and Decoder are **separate programs** that communicate only through
files — so the mapping between emails and integers must itself be saved to a file.
This pattern (encode → black box → decode) is everywhere: ML pipelines call it
*label encoding*, and you already met its purest form in the Information Theory chapter.

---
## Part 1 — Reading the Data

First, create the input data file. Run this provided cell as-is.

In [ ]:
OUR_DATA = """\
sandeep,1, 1.0
nitin, 2, 3.0
subodh, 2, 1.0
sandeep, 2, 2.0
nitin, 1, 2.0
"""

with open("our_data.csv", "w") as f:
    f.write(OUR_DATA)

print(open("our_data.csv").read())

### Exercise 1.1 — Parse One Line

The lines are messy: spaces after commas, a newline at the end. Write
`parse_line(line)` that returns a clean tuple `(user, product_id, score)`
with the right types: `str`, `int`, `float`.

```python
parse_line("sandeep, 2, 2.0\n")   # ("sandeep", 2, 2.0)
parse_line("nitin,1, 3.5")        # ("nitin", 1, 3.5)
```

**Write your solution in `exercises/ex_1_1_parse_line.py`**

In [ ]:
def parse_line(line):
    pass  # replace this


assert parse_line("sandeep, 2, 2.0\n") == ("sandeep", 2, 2.0)
assert parse_line("nitin,1, 3.5")      == ("nitin", 1, 3.5)
assert parse_line("a@b.com , 7 , 0.5") == ("a@b.com", 7, 0.5)
print("parse_line: OK")

### Exercise 1.2 — Read the Whole File

Write `read_data(filename)` that opens the file and returns a list of parsed
tuples, skipping any blank lines.

```python
read_data("our_data.csv")
# [("sandeep", 1, 1.0), ("nitin", 2, 3.0), ("subodh", 2, 1.0),
#  ("sandeep", 2, 2.0), ("nitin", 1, 2.0)]
```

**Write your solution in `exercises/ex_1_2_read_data.py`**

In [ ]:
def read_data(filename):
    pass  # replace this


rows = read_data("our_data.csv")
assert rows == [("sandeep", 1, 1.0), ("nitin", 2, 3.0), ("subodh", 2, 1.0),
                ("sandeep", 2, 2.0), ("nitin", 1, 2.0)]
print("read_data: OK")

---
## Part 2 — The Encoder

### Exercise 2.1 — Assign an Integer to Each Email

Write `build_mapping(rows)` that gives each **distinct** user an integer ID:
1 for the first user seen, 2 for the second new user, and so on.
Users repeat in the data — they must keep the same ID.

```python
build_mapping(rows)   # {"sandeep": 1, "nitin": 2, "subodh": 3}
```

**Write your solution in `exercises/ex_2_1_build_mapping.py`**

In [ ]:
def build_mapping(rows):
    pass  # replace this


mapping = build_mapping(rows)
assert mapping == {"sandeep": 1, "nitin": 2, "subodh": 3}
print("build_mapping: OK")

### Exercise 2.2 — Persist the Mapping

The Decoder is a *separate program* — it will not share memory with the Encoder.
The mapping must survive as a file.

Write `save_mapping(mapping, filename)` (one `email,id` line per user) and
`load_mapping(filename)` that reads it back — **types intact** (ids are ints).

The test is the round trip:

```python
save_mapping(mapping, "mapping.csv")
load_mapping("mapping.csv") == mapping   # True
```

**Write your solution in `exercises/ex_2_2_save_mapping.py`**

In [ ]:
def save_mapping(mapping, filename):
    pass  # replace this


def load_mapping(filename):
    pass  # replace this


save_mapping(mapping, "mapping.csv")
print(open("mapping.csv").read())
assert load_mapping("mapping.csv") == mapping
print("save_mapping / load_mapping: OK")

### Exercise 2.3 — The Complete Encoder Program

Put it together. Write `encode_file(input_file, encoded_file, mapping_file)` that:

1. Reads the raw data (`read_data`),
2. Builds and saves the mapping,
3. Writes the encoded CSV: same rows, emails replaced by their integer IDs.

Expected `encoded.csv` for our data:

```
1,1,1.0
2,2,3.0
3,2,1.0
1,2,2.0
2,1,2.0
```

**Write your solution in `exercises/ex_2_3_encode_file.py`**

In [ ]:
def encode_file(input_file, encoded_file, mapping_file):
    pass  # replace this


encode_file("our_data.csv", "encoded.csv", "mapping.csv")
print(open("encoded.csv").read())

expected = "1,1,1.0\n2,2,3.0\n3,2,1.0\n1,2,2.0\n2,1,2.0\n"
assert open("encoded.csv").read() == expected
assert load_mapping("mapping.csv") == {"sandeep": 1, "nitin": 2, "subodh": 3}
print("encode_file: OK")

---
## Part 3 — The Recommendation Engine (Black Box)

Here is the engine you bought. **You are not allowed to modify it** — run the
cell as-is. For each user it recommends the products they have *not* rated yet,
scored by that product's average rating from other users.

Notice it would crash on emails: it calls `int()` on the first column.
That is exactly why you wrote the Encoder.

In [ ]:
def run_recommender(input_file, output_file):
    """BLACK BOX — the engine you bought. Do not modify."""
    ratings = []
    for line in open(input_file):
        if line.strip():
            u, p, s = line.strip().split(",")
            ratings.append((int(u), int(p), float(s)))

    product_scores = {}
    for u, p, s in ratings:
        product_scores.setdefault(p, []).append(s)
    avg = {p: sum(v) / len(v) for p, v in product_scores.items()}

    seen = {}
    for u, p, s in ratings:
        seen.setdefault(u, set()).add(p)

    with open(output_file, "w") as f:
        for u in sorted(seen):
            for p in sorted(avg):
                if p not in seen[u]:
                    f.write(f"{u},{p},{avg[p]}\n")


run_recommender("encoded.csv", "recommendations.csv")
print(open("recommendations.csv").read())

Only one line came out: user `3` should look at product `1`.
Makes sense — everyone else has already rated everything. But **who is user 3?**
Time for the Decoder.

---
## Part 4 — The Decoder

### Exercise 4.1 — Invert the Mapping

The mapping file maps email → id. The decoder needs id → email.

Write `invert(mapping)`. (You built this in the Dictionaries chapter — rebuild
it from memory.) Why is inversion safe here — what property must the mapping have?

```python
invert({"sandeep": 1, "nitin": 2})   # {1: "sandeep", 2: "nitin"}
```

**Write your solution in `exercises/ex_4_1_invert.py`**

In [ ]:
def invert(mapping):
    pass  # replace this


assert invert({"sandeep": 1, "nitin": 2}) == {1: "sandeep", 2: "nitin"}
print("invert: OK")

### Exercise 4.2 — The Complete Decoder Program

Write `decode_file(recs_file, mapping_file, output_file)` that:

1. Loads the mapping from its file and inverts it,
2. Reads the engine's recommendations,
3. Writes the final CSV with integer IDs translated back to emails.

```python
decode_file("recommendations.csv", "mapping.csv", "final.csv")
open("final.csv").read()   # "subodh,1,1.5\n"
```

So the mystery user 3 was **subodh**, and the pipeline recommends product 1
with score 1.5.

**Write your solution in `exercises/ex_4_2_decode_file.py`**

In [ ]:
def decode_file(recs_file, mapping_file, output_file):
    pass  # replace this


decode_file("recommendations.csv", "mapping.csv", "final.csv")
result = open("final.csv").read()
print(result)
assert result == "subodh,1,1.5\n"
print("decode_file: OK")

---
## Part 5 — Prove the Pipeline

### Exercise 5.1 — Round Trip on Bigger Data

A pipeline you have run once works once. A pipeline you have **tested** works.

The provided cell generates a bigger random dataset. Run your whole pipeline on
it, then write `check_pipeline(original_file, final_file)` that verifies:

1. Every user in the final output existed in the original data (no invented users).
2. No integer IDs leaked into the final output (every user field contains `"@"`).

**Think about:** what could silently break if two different emails ever received
the same ID? Which of your functions guarantees that cannot happen?

**Write your solution in `exercises/ex_5_1_check_pipeline.py`**

In [ ]:
import random

random.seed(7)
users = [f"user{i}@example.com" for i in range(1, 21)]
with open("big_data.csv", "w") as f:
    for _ in range(100):
        u = random.choice(users)
        p = random.randint(1, 10)
        s = round(random.uniform(0.5, 5.0), 1)
        f.write(f"{u}, {p}, {s}\n")

# run YOUR pipeline:
encode_file("big_data.csv", "big_encoded.csv", "big_mapping.csv")
run_recommender("big_encoded.csv", "big_recs.csv")
decode_file("big_recs.csv", "big_mapping.csv", "big_final.csv")

print(f"{len(open('big_recs.csv').readlines())} recommendations generated")
print("first 5 lines of the final output:")
for line in open("big_final.csv").readlines()[:5]:
    print("  ", line, end="")

In [ ]:
def check_pipeline(original_file, final_file):
    pass  # replace this: assert the two properties, return True if all good


assert check_pipeline("big_data.csv", "big_final.csv") == True
print("check_pipeline: OK — the pipeline is verified end to end")

---
## Summary

| Piece | What you built | The general pattern |
|-------|---------------|--------------------|
| `parse_line`, `read_data` | messy text → typed tuples | every pipeline starts with parsing |
| `build_mapping` | email → int, first-seen order | **label encoding** — `sklearn.preprocessing.LabelEncoder` is exactly this |
| `save_mapping`, `load_mapping` | mapping as a file | separate programs share state only through files |
| `encode_file` | the Encoder program | adapt *your* data to *their* interface |
| `run_recommender` | untouched black box | you wrapped it instead of changing it |
| `invert`, `decode_file` | the Decoder program | translate answers back into your domain |
| `check_pipeline` | end-to-end verification | trust pipelines you have tested, not pipelines that ran |

The same encode → black box → decode shape appears when you one-hot-encode
categories for a model, tokenize text for an LLM, or compress data before a
network hop (your Huffman coder!). The black box changes; the wrapper pattern never does.